# Fund Generation

## Objective

This notebook generates a synthetic but realistic simulation of investment fund portfolios.

The goal is to approximate real-world asset management structures by:
- Creating multiple investment funds with varying assets under management (AUM)
- Allocating capital across a diversified universe of financial assets
- Using probabilistic weighting (Dirichlet distribution) to simulate realistic portfolio concentration
- Producing a dataset suitable for risk modeling and machine learning pipelines

Although the underlying assets are real (Yahoo Finance tickers), the fund compositions and allocations are artificially generated and do not represent actual investment strategies or real portfolios.

This dataset is designed for:
- Risk modeling (VaR, liquidity risk, stress testing)
- Feature engineering for ML models
- End-to-end simulation of a Data + AI pipeline

---

## Imports


In [0]:
import numpy as np
import pandas as pd

## Asset Universe (50 tickers from Yahoo Finance)

In [0]:
def get_asset_universe():
    """
    Returns a list of 50 real Yahoo Finance tickers.

    Args:
        None

    Returns:
        list: List of tickers
    """

    tickers = [
        # US Tech
        "AAPL", "MSFT", "AMZN", "GOOGL", "META",
        "NVDA", "TSLA", "NFLX", "AMD", "INTC",
        "ORCL", "CRM", "IBM", "QCOM", "AVGO",
        "ADBE", "CSCO",

        # ETFs
        "SPY", "QQQ", "VTI", "IWM", "DIA", "VOO",

        # Financials
        "JPM", "BAC", "WFC", "GS", "MS",

        # Healthcare
        "JNJ", "PFE", "UNH", "MRK", "ABBV",

        # Consumer
        "PG", "KO", "PEP", "NKE", "MCD", "SBUX",

        # Energy
        "XOM", "CVX",

        # Others
        "DIS", "V",

        # Brazil
        "PETR4.SA", "VALE3.SA", "ITUB4.SA",
        "WEGE3.SA", "B3SA3.SA", "ABEV3.SA", "BBAS3.SA"
    ]

    return tickers

## Generate Funds

In [0]:
def generate_funds(n_funds=8, seed=42):
    """
    Generate realistic fund portfolios.

    Each fund:
    - Random AUM between 5M and 25M
    - Random number of assets (5, 8, 15)
    - Dirichlet weights (sum = 1)

    Args:    
        n_funds (int): Number of funds to generate
        seed (int): Random seed

    Returns:
        pd.DataFrame: DataFrame with columns ['fund_id', 'fund_pl', 'ticker', 'weight_pct', 'position_value']
    """

    np.random.seed(seed)
    tickers = get_asset_universe()
    portfolio_sizes = [5, 8, 15]
    records = []

    for fund_id in range(1, n_funds + 1):

        fund_name = f"FUND_{fund_id:03d}"
        fund_pl = round(np.random.uniform(5_000_000, 25_000_000), 2)
        k = np.random.choice(portfolio_sizes)
        selected_assets = np.random.choice(
            tickers,
            size=k,
            replace=False
        )

        weights = np.random.dirichlet(np.ones(k))

        for ticker, weight in zip(selected_assets, weights):
            records.append({
                "fund_id": fund_name,
                "fund_pl": fund_pl,
                "ticker": ticker,
                "weight_pct": float(weight),
                "position_value": float(fund_pl * weight)
            })

    df = pd.DataFrame(records)

    return df

## Convert to Spark

In [0]:
def to_spark(df):
    """
    Convert a pandas DataFrame to a Spark DataFrame.

    Args:
        df (pd.DataFrame): Input pandas DataFrame.

    Returns:
        pyspark.sql.DataFrame: Converted Spark DataFrame.
    """

    df = spark.createDataFrame(df)

    return df

## Save Data

In [0]:
def save_data(df, table_name):
    """
    Save data to a Delta table.

    Args:
        df (DataFrame): Spark DataFrame
        table_name (str): Name of the Delta table

    Returns:
        None
    """

    df.write.format("delta") \
        .mode('overwrite') \
        .option("overwriteSchema", "true") \
        .option("mergeSchema", "true") \
        .saveAsTable(table_name)

## Run Pipeline

In [0]:
def main():
    df = generate_funds(n_funds=8)
    spark_df = to_spark(df)
    save_data(spark_df, "risk_management.funds")

In [0]:
if __name__ == "__main__":
    main()